In [81]:
import csv
import math
import random
import urllib.request

In [82]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv",
    "iris.data"
)
print("iris.data downloaded successfully.")

iris.data downloaded successfully.


In [83]:
# STEP 1 : LOAD DATASET FROM CSV FILE
def load_dataset():
    dataset = []
    with open('iris.data', 'r') as f:
        reader = csv.reader(f)  #data row by row read korbe
        for row in reader:
            if len(row) == 5:
                features = [float(row[0]), float(row[1]), float(row[2]), float(row[3])]
                label = row[4]
                dataset.append(features + [label])
    print(f"Dataset loaded: {len(dataset)} samples")
    print(f"  Classes : Iris-setosa, Iris-versicolor, Iris-virginica")
    print(f"  Features: sepal_length, sepal_width, petal_length, petal_width")
    return dataset

In [84]:
# STEP 2 — SPLIT DATASET  (70% train | 15% val | 15% test)

def split_dataset(dataset, seed=42):
    random.seed(seed)

    train_set = []
    val_set   = []
    test_set  = []

    for sample in dataset:
        R = random.random()
        if R <= 0.70:
            train_set.append(sample)
        elif R <= 0.85:
            val_set.append(sample)
        else:
            test_set.append(sample)

    print(f"\n--- Dataset Split (seed={seed}) ---")
    print(f"  Training   : {len(train_set)} samples")
    print(f"  Validation : {len(val_set)} samples")
    print(f"  Test       : {len(test_set)} samples")
    return train_set, val_set, test_set


In [85]:
# STEP 3 — KNN ALGORITHM

def euclidean_distance(a, b):
    total = 0.0
    for i in range(4):
        total += (a[i] - b[i]) ** 2
    return math.sqrt(total)


def majority_vote(neighbours):
    counts = {}
    for neighbour in neighbours:
        label = neighbour[4]                        # index 4 = class label
        counts[label] = counts.get(label, 0) + 1
    return max(counts, key=counts.get)              # class with highest count

def get_distance(pair):
    return pair[1]

def knn_predict(train_set, query, k):
    distances = []
    for train_sample in train_set:
        d = euclidean_distance(query, train_sample)
        distances.append((train_sample, d))

    distances.sort(key=get_distance)

    k_nearest = [pair[0] for pair in distances[:k]]

    return majority_vote(k_nearest)

In [86]:
# STEP 4 — ACCURACY EVALUATION

def evaluate(train_set, eval_set, k):
    correct = 0
    for sample in eval_set:
        true_label = sample[4] # actual class
        predicted_label = knn_predict(train_set, sample, k)
        if predicted_label == true_label:
            correct += 1
    return (correct / len(eval_set)) * 100


In [87]:
# STEP 5 — K TUNING ON VALIDATION SET

def tune_k(train_set, val_set, k_values):

    results= {}
    best_k= None
    best_accuracy = -1

    for k in k_values:
        acc = evaluate(train_set, val_set, k)
        results[k] = acc

        if acc > best_accuracy:
            best_accuracy = acc
            best_k = k

        print(f"  k = {k:<10} {acc:>18.2f}%")

    print(f"\n  Best k = {best_k}  |  Validation Accuracy = {best_accuracy:.2f}%")
    return results, best_k

In [88]:
# STEP 6 — FINAL TEST SET ACCURACY (using best k only)

def final_test(train_set, test_set, best_k):
    test_acc = evaluate(train_set, test_set, best_k)
    print("   Final Test Set Evaluation")
    print(f"  Best k used   : {best_k}")
    print(f"  Test Accuracy : {test_acc:.2f}%")
    return test_acc

In [89]:
dataset = load_dataset()
train_set, val_set, test_set = split_dataset(dataset, seed=42)
results, best_k = tune_k(train_set, val_set, k_values=[1, 3, 5, 10, 15])
final_test(train_set, test_set, best_k)

Dataset loaded: 150 samples
  Classes : Iris-setosa, Iris-versicolor, Iris-virginica
  Features: sepal_length, sepal_width, petal_length, petal_width

--- Dataset Split (seed=42) ---
  Training   : 108 samples
  Validation : 19 samples
  Test       : 23 samples
  k = 1                       94.74%
  k = 3                      100.00%
  k = 5                      100.00%
  k = 10                      94.74%
  k = 15                      94.74%

  Best k = 3  |  Validation Accuracy = 100.00%
   Final Test Set Evaluation
  Best k used   : 3
  Test Accuracy : 91.30%


91.30434782608695